# NutriScan — training the food recognition model

Run this notebook in **Google Colab** with a GPU: *Runtime → Change runtime type → T4 GPU*.

What it does:
1. Downloads the **Food-101** dataset and keeps the 20 dishes our app knows.
2. Fine-tunes a **ResNet-18** pretrained on ImageNet (transfer learning).
3. Shows loss/accuracy curves and a confusion matrix (put them on your slides).
4. Exports `food_model.pth` + `classes.json` → copy them to `backend/ml/weights/`.

⚠️ The model architecture and the test preprocessing here must stay the same as in `backend/ml/predictor.py`.

In [ ]:
import json, time
import torch, torchvision
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)

## 1. Settings
These labels must match `Dish.ml_label` in the database (see `seed_demo.py`).

In [ ]:
CLASSES = [
    'pizza', 'hamburger', 'french_fries', 'hot_dog', 'sushi', 'ramen', 'fried_rice',
    'dumplings', 'spaghetti_bolognese', 'chicken_curry', 'steak', 'caesar_salad',
    'greek_salad', 'pancakes', 'omelette', 'waffles', 'ice_cream', 'chocolate_cake',
    'cheesecake', 'donuts',
]
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 5          # 5 is enough for a demo; try 8-10 for better accuracy
LR = 1e-3
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet values

## 2. Data
Food-101 is ~5 GB, the first download takes a few minutes.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# Same as backend/ml/predictor.py - do not change one without the other
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class SelectedFood(Dataset):
    """Food-101 limited to our CLASSES, with labels renumbered 0..len(CLASSES)-1."""
    def __init__(self, split, transform):
        self.base = datasets.Food101('data', split=split, download=True)
        self.transform = transform
        old_to_new = {self.base.classes.index(c): i for i, c in enumerate(CLASSES)}
        self.items = [(idx, old_to_new[lbl]) for idx, lbl in enumerate(self.base._labels) if lbl in old_to_new]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        idx, label = self.items[i]
        image, _ = self.base[idx]
        return self.transform(image), label

train_ds = SelectedFood('train', train_tf)
test_ds = SelectedFood('test', test_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
print('train images:', len(train_ds), '| test images:', len(test_ds))

## 3. Model: pretrained ResNet-18 with a new last layer

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))  # same as predictor.py
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 4. Training

In [ ]:
def evaluate():
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for x, y in test_dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total

history = {'train_loss': [], 'test_loss': [], 'test_acc': []}
for epoch in range(EPOCHS):
    model.train()
    start, loss_sum, n = time.time(), 0.0, 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        n += len(y)
    scheduler.step()
    test_loss, test_acc = evaluate()
    history['train_loss'].append(loss_sum / n)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    print(f'epoch {epoch + 1}/{EPOCHS}  train_loss {loss_sum / n:.3f}  test_loss {test_loss:.3f}  test_acc {test_acc:.1%}  ({time.time() - start:.0f}s)')

## 5. Results for the slides

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['test_loss'], label='test')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history['test_acc']); ax[1].set_title('Test accuracy'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
model.eval()
preds, labels = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        labels += y.tolist()
print(classification_report(labels, preds, target_names=CLASSES, digits=3))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(labels, preds, display_labels=CLASSES, xticks_rotation=90, ax=ax, colorbar=False)
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

## 6. Export for the Django backend
Download both files and put them in `backend/ml/weights/`.

In [ ]:
model.cpu()
torch.save(model.state_dict(), 'food_model.pth')
with open('classes.json', 'w') as f:
    json.dump(CLASSES, f)

from google.colab import files
for name in ['food_model.pth', 'classes.json', 'training_curves.png', 'confusion_matrix.png']:
    files.download(name)